In [6]:
import pandas as pd, numpy as np

SRC = r"D:\data analysis\VOLTIX\task9\Customers_Fakedata.csv"
OUT = r"D:\data analysis\VOLTIX\task9\Customers_cleaned.csv"

df = pd.read_csv(SRC)
df = df.drop_duplicates()

df['Name'] = df['Name'].str.strip().str.title()
df['Email'] = df['Email'].str.strip().str.lower()
df['First'] = df['Name'].str.split().str[0]

In [7]:
# Gender: standardize, then fill missing from first name
df['Gender'] = df['Gender'].str.strip().str.lower().map(
    {'m': 'Male', 'male': 'Male', 'f': 'Female', 'female': 'Female'})
name_gender = {'Ahmed': 'Male', 'Ali': 'Male', 'John': 'Male', 'Mark': 'Male',
               'Fatma': 'Female', 'Lina': 'Female', 'Sara': 'Female',
               'Alaa': 'Male'}
df['Gender'] = df['Gender'].fillna(df['First'].map(name_gender))

In [8]:
# Age: -1 / 200 / missing -> random values around the mean, between 18 and 90
rng = np.random.default_rng(42)
valid = df['Age'].where(df['Age'].between(10, 100))
mu, sd = valid.mean(), valid.std()
bad = valid.isna()
df['Age'] = valid.astype(float)
df.loc[bad, 'Age'] = np.clip(np.round(rng.normal(mu, sd, bad.sum())), 18, 90)
df['Age'] = df['Age'].astype(int)
df['AgeGroup'] = pd.cut(df['Age'], [0, 24, 34, 44, 54, 64, 100],
                        labels=['<25', '25-34', '35-44', '45-54', '55-64', '65+']).astype(str)

In [9]:
# Rating (valid 1-5), Amount, Category: leave missing empty
df['Rating'] = df['Rating'].where(df['Rating'].between(1, 5))
df['PurchaseAmount'] = df['PurchaseAmount'].where(df['PurchaseAmount'] > 0)

# Dates (invalid -> empty)
df['PurchaseDate'] = pd.to_datetime(df['PurchaseDate'], format='%m/%d/%Y', errors='coerce')
df['Year'] = df['PurchaseDate'].dt.year
df['Month'] = df['PurchaseDate'].dt.to_period('M').astype(str).replace('NaT', np.nan)

In [10]:
df = df.drop(columns=['Phone', 'First'])
df.to_csv(OUT, index=False)

print(df.shape)
print(df.isna().sum())
df.head()

(2100, 12)
CustomerID           0
Name                 0
Age                  0
Gender               0
Email                0
PurchaseAmount      97
PurchaseDate       118
ProductCategory    565
Rating             613
AgeGroup             0
Year               118
Month              118
dtype: int64


,CustomerID,Name,Age,Gender,Email,PurchaseAmount,PurchaseDate,ProductCategory,Rating,AgeGroup,Year,Month
0,CUST1000,Ali Hassan,72,Male,ali.hassan@gmail.com,837.31,2023-10-12,Books,3.0,65+,2023.0,2023-10
1,CUST1001,Fatma Ali,61,Female,fatma.ali@hotmail.com,900.25,2023-02-23,NaN,3.0,55-64,2023.0,2023-02
2,CUST1002,Ahmed Mahmoud,32,Male,ahmed.mahmoud@hotmail.com,761.72,2023-07-07,Toys,2.0,25-34,2023.0,2023-07
3,CUST1003,Ahmed Gaber,71,Female,ahmed.gaber@gmail.com,303.72,2023-12-30,NaN,2.0,65+,2023.0,2023-12
4,CUST1004,Ali Ibrahim,73,Male,ali.ibrahim@yahoo.com,635.81,2023-12-07,Clothing,2.0,65+,2023.0,2023-12
